# 01 — Sanity check

End-to-end verification before running the full compression sweep:

1. GPU available
2. Drive mounted, repo accessible
3. At least one checkpoint loads correctly
4. Forward pass produces sensible logits
5. Baseline accuracy on ImageNet-100 val is in expected range (>0.6)
6. Magnitude pruning at small ratio degrades accuracy gracefully

If all six pass, the full sweep can be launched in `02_compression_main_workflow.ipynb`.

---

## 🔧 Configuration required

Before running, you must set the following paths in this notebook:

- **Cell 3** — `REPO_SRC` *(only if you cloned the repo to a custom Drive location; otherwise leave default)*
- **Cell 5** — `CHECKPOINT_ROOT`: folder on your Drive holding the 12 pretrained checkpoints (download link in the repository README; layout `{pe_type}_seed{seed}/best_model.pth`)
- **Cell 7** — `VAL_ROOT`: where ImageNet-100 val will be extracted to (typically the ephemeral `/content/imagenet100/val`; only change if you have a persistent copy)

All other paths use ephemeral `/content/` storage and do **not** need to be changed.

## 1. GPU + Drive setup

In [ ]:
# 1.1 GPU check  (no changes needed)
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('Memory: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))

In [ ]:
# 1.2 Mount Drive  (no changes needed)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1.3 Bring the repo into /content
# ===========================================================================
# >>> USER CONFIGURATION (optional) <<<
# Set REPO_SRC to the folder on YOUR Google Drive where you unpacked
# this repository's code. The default below matches the layout used
# in the workflow documentation.
# ===========================================================================
import shutil, os
REPO_SRC = '/content/drive/MyDrive/pe_compression_experiment/code'  # <-- EDIT if needed

# ---------------------------------------------------------------------------
# Destination on the Colab instance  (no changes needed)
# ---------------------------------------------------------------------------
REPO_DST = '/content/vit-pe-compression'
if os.path.exists(REPO_DST):
    shutil.rmtree(REPO_DST)
shutil.copytree(REPO_SRC, REPO_DST)
%cd /content/vit-pe-compression

## 2. Check checkpoints are visible

In [ ]:
# 2.1 List available checkpoints
# ===========================================================================
# >>> USER CONFIGURATION REQUIRED <<<
# Set CHECKPOINT_ROOT to the folder on YOUR Google Drive where you placed
# the 12 pretrained checkpoints. Expected layout:
#     <CHECKPOINT_ROOT>/learned_seed42/best_model.pth
#     <CHECKPOINT_ROOT>/learned_seed123/best_model.pth
#     ... (12 total)
# ===========================================================================
from models import list_available_checkpoints

CHECKPOINT_ROOT = '/content/drive/MyDrive/Trained models_ImageNet100'  # <-- EDIT THIS

available = list_available_checkpoints(CHECKPOINT_ROOT)
print(f'Found {len(available)} checkpoints (expected: 12):')
for pe, seed in available:
    print(f'  {pe}_seed{seed}')

## 3. Load one model + forward pass

In [ ]:
# 3.1 Load model and run a forward pass on dummy input  (no changes needed)
from models import load_pretrained_model

model = load_pretrained_model(CHECKPOINT_ROOT, 'rope', 42, device='cuda')
print('Model loaded:', type(model).__name__)
print('PE type:', model.pe_type)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {n_params/1e6:.2f}M')

x = torch.randn(2, 3, 224, 224, device='cuda')
with torch.no_grad():
    logits = model(x)
print('Logits shape:', logits.shape)

## 4. ImageNet-100 val loader sanity

In [ ]:
# 4.1 Load val set
# ===========================================================================
# >>> USER CONFIGURATION <<<
# VAL_ROOT is where the ImageNet-100 val set lives on the Colab instance.
# The default (/content/imagenet100/val) is ephemeral and gets created by
# the setup_imagenet100_val.py script in 02_compression_main_workflow.ipynb.
# If you have a persistent copy on Drive, point this at it instead.
# ===========================================================================
from data import get_imagenet100_val_loader

VAL_ROOT = '/content/imagenet100/val'  # <-- EDIT only if using a persistent copy

loader, dataset = get_imagenet100_val_loader(VAL_ROOT, batch_size=64, num_workers=2)
print(f'Val set: {len(dataset)} images, {len(dataset.classes)} classes')

x, y = next(iter(loader))
print(f'Batch: x={tuple(x.shape)}, y={tuple(y.shape)}, y in [{y.min().item()}, {y.max().item()}]')

## 5. Quick baseline accuracy (first 5 batches only)

In [ ]:
# 5.1 Quick accuracy check  (no changes needed)
from tqdm import tqdm

correct, total = 0, 0
with torch.no_grad():
    for i, (x, y) in enumerate(tqdm(loader, total=5)):
        if i >= 5: break
        x, y = x.cuda(), y.cuda()
        pred = model(x).argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
print(f'Top-1 accuracy (first 5 batches, {total} images): {correct/total:.4f}')
print('Expected: > 0.6 for a well-trained ImageNet-100 ViT-Base')

## 6. Magnitude pruning sanity (small ratio)

In [ ]:
# 6.1 Apply pruning at small ratios and check graceful degradation  (no changes needed)
import copy
from scripts.magnitude_pruning import apply_pruning, evaluate

original = copy.deepcopy(model.state_dict())

for ratio in [0.0, 0.1, 0.3]:
    model.load_state_dict(original)
    stats = apply_pruning(model, scope='global', ratio=ratio)
    # quick accuracy on first 10 batches
    correct, total = 0, 0
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if i >= 10: break
            x, y = x.cuda(), y.cuda()
            pred = model(x).argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    print(f'ratio={ratio:.1f}  sparsity={stats["actual_sparsity"]:.3f}  acc(quick)={correct/total:.4f}')

If all six cells produced sensible output, the pipeline is healthy.
Move to `02_main_workflow.ipynb` for the full sweep.